# 03. Target Definition, Leakage Prevention & Feature Importance

Formulation of the 24-hour event-centred early warning target, class imbalance characterization, multi-machine leakage prevention, and Random Forest feature importance selection.


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from src.data.load_data import load_workbook
from src.targets.build_target import build_early_warning_target
from src.utils.io import load_config

sns.set_theme(style="whitegrid")
config = load_config()
workbook = load_workbook(config["project"]["raw_file"])
telemetry = workbook["telemetry"]
events = workbook["events"]


## 1. Multi-Horizon Early Warning Target Comparison


In [ ]:
horizons = [6, 12, 24, 48, 72]
summary = []

for h in horizons:
    t = build_early_warning_target(telemetry, events, horizon_hours=h)
    pos = int(t.sum())
    total = len(t)
    summary.append({
        "Horizon (Hours)": h,
        "Positive Hours": pos,
        "Total Hours": total,
        "Positive Class (%)": (pos / total) * 100
    })

horizon_df = pd.DataFrame(summary)
display(horizon_df)

plt.figure(figsize=(8, 4))
sns.barplot(data=horizon_df, x="Horizon (Hours)", y="Positive Class (%)", palette="crest")
plt.title("Positive Target Class Ratio across Early Warning Horizons", fontsize=14)
plt.tight_layout()
plt.show()


## 2. 24-Hour Target Class Imbalance Breakdown


In [ ]:
df_target = telemetry.copy()
df_target["target_failure"] = build_early_warning_target(telemetry, events, horizon_hours=24)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
df_target["target_failure"].value_counts().plot.pie(
    autopct="%1.2f%%", colors=["lightblue", "lightcoral"], ax=ax1, startangle=90, wedgeprops={"edgecolor": "black"}
)
ax1.set_title("24h Early Warning Class Imbalance", fontsize=13)
ax1.set_ylabel("")

sns.countplot(data=df_target, x="Machine ID", hue="target_failure", palette=["steelblue", "crimson"], ax=ax2)
ax2.set_title("Target Distribution by Dozer", fontsize=13)
plt.tight_layout()
plt.show()

print("Target Counts:")
print(df_target["target_failure"].value_counts())


## 3. Degradation Feature Importance Ranking


In [ ]:
df_features = pd.read_csv(config["project"]["featured_file"], parse_dates=["Timestamp"])
X = df_features.select_dtypes(include=np.number).drop(columns=["target_failure"], errors="ignore")
y = df_features["target_failure"].astype(int)

X_clean = X.fillna(X.median())

rf = RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=42)
rf.fit(X_clean, y)

feat_imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print("Top 15 Most Discriminative Degradation Features:")
print(feat_imp.head(15))

plt.figure(figsize=(10, 6))
feat_imp.head(15).plot(kind="bar", color="skyblue", edgecolor="black")
plt.title("Top 15 Degradation & Threshold Features (Random Forest Importance)", fontsize=13)
plt.ylabel("Importance Score")
plt.xlabel("Engineered Features")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()
